# Joined-station EDA: relationships across the river network

This notebook explores the value the joined multi-station dataset adds beyond a
single-station view: how the target gauge (Korneuburg, `207241-at`) relates to the
other stations retained by `02_preprocessing.ipynb` on its timeline.

It asks three questions:

1. How complete is each station's contribution to the target's timeline?
2. How strongly, and at what lag, does each station's water level track the target's?
3. Does the distance-based "upstream" ordering used to build the join actually agree
   with what the data shows?

All train and test rows are concatenated in timestamp order and treated as one
continuous history. The single internal generation seam between them is marked as a
reference line on time-indexed plots only; there is no dedicated boundary-effect
section, since the base joined data used here carries no lag or rolling features to
examine near that seam.

**Units.** Water level is centimetres (cm), precipitation is millimetres per hour
(mm), and temperature is degrees Celsius (°C). All timestamps use UTC.

> **Interpretation caveat:** combining the complete history is appropriate for
> descriptive EDA, but results from this notebook are not blind test evidence and
> must not be reported as out-of-sample model performance.

## 1. Configuration

Station identifiers and paths are local to this notebook, except for the station
list, which is imported from `src.config` so this notebook cannot drift from the
real pipeline's configuration. The notebook is read-only with respect to every
source artifact.

In [ ]:
from __future__ import annotations

import hashlib
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display
from plotly.subplots import make_subplots

from src.config import STATION_IDS, TARGET_STATION_ID
from src.feature_engineering import extract_station_frame

warnings.filterwarnings(
    "ignore",
    message="The 'generic' unit for NumPy timedelta is deprecated",
    category=DeprecationWarning,
)

RAW_DIR = Path("data/raw")
JOINED_DIR = Path("data/processed/joined")
LAG_SCAN_HOURS = 72
WEAK_CORRELATION_THRESHOLD = 0.5
RANK_GAP_FLAG = 3

PATHS = {
    "station_catalog": RAW_DIR / "pegelalarm_stations_at.parquet",
    "preprocess_manifest": JOINED_DIR / "all_stations_preprocess_metadata.json",
}

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

PLOT_TEMPLATE = "plotly_white"
STATION_PALETTE = [
    "#176B87",
    "#3FA7D6",
    "#D95F59",
    "#F2A900",
    "#7A5195",
    "#0B3C5D",
    "#4C956C",
    "#9C6644",
    "#6C757D",
    "#845EC2",
]
STATION_COLORS = dict(zip(STATION_IDS, STATION_PALETTE, strict=True))
SEAM_COLOR = "#B22222"

## 2. Load and validate the joined preprocessing artifact

The checks below fail fast on missing files, hash drift, schema drift, or a
timeline that is not contiguous hourly UTC. No per-station extraction is needed
here: the joined frame itself, at its full joined-schema width, is the object of
study for the rest of this notebook.


In [ ]:
def sha256(path: Path) -> str:
    """Return the SHA-256 digest for a file without modifying it."""
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def utc_text(value: pd.Timestamp) -> str:
    """Format a timezone-aware timestamp in manifest-compatible UTC text."""
    return pd.Timestamp(value).isoformat().replace("+00:00", "Z")


def validate_profile(profile: dict, frame: pd.DataFrame) -> None:
    """Validate a loaded frame against one manifest profile."""
    path = Path(profile["path"])
    assert path.is_file()
    assert sha256(path) == profile["sha256"]
    assert len(frame) == profile["rows"]
    assert {column: str(dtype) for column, dtype in frame.dtypes.items()} == (
        profile["schema"]
    )
    assert (
        utc_text(frame["timestamp"].iloc[0]) == profile["timestamp_range"]["start_utc"]
    )
    assert (
        utc_text(frame["timestamp"].iloc[-1]) == profile["timestamp_range"]["end_utc"]
    )


for path in PATHS.values():
    assert path.is_file(), f"Required input is missing: {path}"

preprocess_manifest = json.loads(
    PATHS["preprocess_manifest"].read_text(encoding="utf-8")
)
station_catalog = pd.read_parquet(PATHS["station_catalog"])

configured_station_ids = set(STATION_IDS)
manifest_station_ids = set(preprocess_manifest["station_ids"])
assert TARGET_STATION_ID in configured_station_ids
assert TARGET_STATION_ID in manifest_station_ids
assert preprocess_manifest["target_station_id"] == TARGET_STATION_ID
assert manifest_station_ids <= configured_station_ids

preprocess_coverage = preprocess_manifest["coverage"]
retained_station_ids = list(preprocess_manifest["station_ids"])
retained_from_coverage = {
    record["station_id"] for record in preprocess_coverage if record["retained"]
}
assert retained_from_coverage == manifest_station_ids
excluded_station_ids = sorted(configured_station_ids - manifest_station_ids)
reported_excluded_station_ids = [
    record["station_id"] for record in preprocess_coverage if not record["retained"]
]
assert set(reported_excluded_station_ids) == set(excluded_station_ids)
if excluded_station_ids:
    print(
        "Configured stations excluded by the preprocessing overlap filter: "
        + ", ".join(excluded_station_ids)
    )
else:
    print("No configured stations were excluded by the preprocessing overlap filter.")

source_profiles = sorted(
    preprocess_manifest["artifacts"].values(),
    key=lambda profile: profile["timestamp_range"]["start_utc"],
)
source_parts = [pd.read_parquet(profile["path"]) for profile in source_profiles]
for profile, frame in zip(source_profiles, source_parts, strict=True):
    validate_profile(profile, frame)
generator = preprocess_manifest["generator"]
assert sha256(Path(generator["module"])) == generator["sha256"]
assert preprocess_manifest["rows"]["total"] == sum(len(part) for part in source_parts)

seam_timestamp = pd.Timestamp(source_profiles[1]["timestamp_range"]["start_utc"])
joined = (
    pd.concat(source_parts, ignore_index=True)
    .sort_values("timestamp")
    .reset_index(drop=True)
)
assert joined["timestamp"].is_monotonic_increasing
assert not joined["timestamp"].duplicated().any()
expected_grid = pd.date_range(
    joined["timestamp"].iloc[0], periods=len(joined), freq="h"
)
assert pd.DatetimeIndex(joined["timestamp"]).equals(expected_grid)

print(
    f"Loaded and validated {len(joined):,} joined hourly rows "
    f"({joined['timestamp'].iloc[0]} to {joined['timestamp'].iloc[-1]}); "
    f"seam at {seam_timestamp}."
)

## 3. Station catalog context

Stations are ordered by distance from the target gauge along the river, the same
`positionKm`-based ordering `02_preprocessing.ipynb` uses to build the join. This
ordering is a labeling convenience, not a hydrological claim; Section 9 checks it
against the data.

In [ ]:
station_positions = station_catalog.set_index("commonid")["positionKm"]
target_position_km = float(station_positions.loc[TARGET_STATION_ID])


def distance_to_target(station_id: str) -> float:
    """Absolute river-km distance between a station and the target station."""
    return abs(float(station_positions.loc[station_id]) - target_position_km)


station_order = [
    TARGET_STATION_ID,
    *sorted(
        (
            station_id
            for station_id in retained_station_ids
            if station_id != TARGET_STATION_ID
        ),
        key=distance_to_target,
    ),
]
upstream_order = [
    station_id for station_id in station_order if station_id != TARGET_STATION_ID
]

catalog_rows = station_catalog.set_index("commonid").loc[station_order]
station_context = pd.DataFrame(
    {
        "station_id": station_order,
        "name": catalog_rows["stationName"].tolist(),
        "water_body": catalog_rows["water"].tolist(),
        "region": catalog_rows["region"].tolist(),
        "position_km": catalog_rows["positionKm"].tolist(),
        "distance_from_target_km": [distance_to_target(s) for s in station_order],
        "role": [
            "target" if s == TARGET_STATION_ID else "upstream candidate"
            for s in station_order
        ],
    }
)
display(station_context)

## 4. Raw PegelAlarm water-level distributions

These histograms use the raw PegelAlarm `value` column from each retained
station's source parquet file. They deliberately do not use imputed values or
any processed/joined columns.

In [ ]:
raw_level_values = {}
for station_id in station_order:
    raw_path = RAW_DIR / f"pegelalarm_{station_id}_height_hour.parquet"
    assert raw_path.is_file(), f"Required raw input is missing: {raw_path}"
    raw_frame = pd.read_parquet(raw_path)
    assert {"value", "station_id"}.issubset(raw_frame.columns)
    assert raw_frame["station_id"].dropna().eq(station_id).all()
    raw_level_values[station_id] = raw_frame["value"].dropna()

histogram_columns = 2
histogram_rows = (len(station_order) + histogram_columns - 1) // histogram_columns
raw_histogram_figure = make_subplots(
    rows=histogram_rows,
    cols=histogram_columns,
    subplot_titles=station_order,
    vertical_spacing=0.08,
)
for position, station_id in enumerate(station_order):
    row, col = position // histogram_columns + 1, position % histogram_columns + 1
    raw_histogram_figure.add_trace(
        go.Histogram(
            x=raw_level_values[station_id],
            name=station_id,
            nbinsx=60,
            marker_color=STATION_COLORS[station_id],
            showlegend=False,
        ),
        row=row,
        col=col,
    )
for position in range(len(station_order), histogram_rows * histogram_columns):
    row, col = position // histogram_columns + 1, position % histogram_columns + 1
    raw_histogram_figure.update_xaxes(visible=False, row=row, col=col)
    raw_histogram_figure.update_yaxes(visible=False, row=row, col=col)

assert len(raw_histogram_figure.data) == len(station_order)
raw_histogram_figure.update_xaxes(title_text="Raw water level (cm)")
raw_histogram_figure.update_yaxes(title_text="Count")
raw_histogram_figure.update_layout(
    title="Raw PegelAlarm water-level distributions by retained station",
    template=PLOT_TEMPLATE,
    height=max(320, 300 * histogram_rows),
)
raw_histogram_figure.show()

## 5. Per-station coverage and data quality

Coverage is measured against the target's own timeline: for each station, the
share of the target's hourly grid where that station has a matching row, and the
water-level missingness within those matched hours. The monthly timeline view
marks the train/test seam for reference.

In [ ]:
coverage_records = []
for station_id in station_order:
    station = extract_station_frame(joined, station_id)
    coverage_records.append(
        {
            "station_id": station_id,
            "usable_hours": len(station),
            "coverage_pct": 100 * len(station) / len(joined),
            "water_level_missing_pct": (
                100 * station["water_level"].isna().mean() if len(station) else np.nan
            ),
            "precipitation_missing_pct": (
                100 * station["precipitation"].isna().mean() if len(station) else np.nan
            ),
            "temperature_missing_pct": (
                100 * station["temperature_2m"].isna().mean()
                if len(station)
                else np.nan
            ),
        }
    )
coverage = pd.DataFrame(coverage_records)
display(coverage)

coverage_figure = go.Figure()
coverage_figure.add_trace(
    go.Bar(
        x=coverage["station_id"],
        y=coverage["coverage_pct"],
        name="Timeline coverage",
        marker_color=[STATION_COLORS[s] for s in coverage["station_id"]],
    )
)
coverage_figure.add_trace(
    go.Scatter(
        x=coverage["station_id"],
        y=coverage["water_level_missing_pct"],
        mode="markers",
        name="Water-level missing (within coverage)",
        marker={"color": "#B22222", "size": 10, "symbol": "diamond"},
    )
)
coverage_figure.update_layout(
    title="Per-station timeline coverage and within-coverage water-level missingness",
    template=PLOT_TEMPLATE,
    height=430,
    yaxis_title="%",
)
coverage_figure.show()

In [ ]:
month_key = joined["timestamp"].dt.tz_localize(None).dt.to_period("M")
monthly_frames = []
for station_id in station_order:
    prefix = f"{station_id}__"
    water_present = joined[f"{prefix}water_level"].notna()
    monthly = (
        pd.DataFrame({"month": month_key, "water_present": water_present})
        .groupby("month", observed=True)
        .agg(hours=("water_present", "size"), water_hours=("water_present", "sum"))
        .reset_index()
    )
    monthly["station_id"] = station_id
    monthly["water_coverage_pct"] = 100 * monthly["water_hours"] / monthly["hours"]
    monthly["month"] = monthly["month"].dt.to_timestamp().dt.tz_localize("UTC")
    monthly_frames.append(monthly)
monthly_coverage = pd.concat(monthly_frames, ignore_index=True)

coverage_timeline_figure = go.Figure()
for station_id in station_order:
    subset = monthly_coverage.loc[monthly_coverage["station_id"] == station_id]
    coverage_timeline_figure.add_trace(
        go.Scatter(
            x=subset["month"],
            y=subset["water_coverage_pct"],
            mode="lines",
            name=station_id,
            line={"color": STATION_COLORS[station_id]},
        )
    )
coverage_timeline_figure.add_vline(
    x=seam_timestamp,
    line_dash="dash",
    line_color=SEAM_COLOR,
    annotation_text="train/test seam",
)
coverage_timeline_figure.update_layout(
    title="Monthly water-level coverage by station",
    template=PLOT_TEMPLATE,
    height=460,
    yaxis_title="Coverage (%)",
    yaxis_range=[0, 101],
    hovermode="x unified",
)
coverage_timeline_figure.show()

## 6. Completeness, largest gaps, and full-join eligibility

This section measures usable water-level coverage against the complete target
timeline, not just whether a station has any matching row. A usable hour has a
station row and a non-missing `water_level`; imputed water levels count as usable
because they are produced by the preprocessing contract. Missing hours are
classified as an absent station row or a missing water level, and each contiguous
run is classified as leading, internal, or trailing.

The threshold below is the operational definition of enough data for this EDA's
full-join comparison. It is configurable: weather missingness is reported
separately and does not decide water-level full-join eligibility.

In [ ]:
FULL_JOIN_MIN_COVERAGE_PCT = 90.0
assert 0.0 <= FULL_JOIN_MIN_COVERAGE_PCT <= 100.0

target_hours = len(joined)
target_start = joined["timestamp"].iloc[0]
target_end = joined["timestamp"].iloc[-1]

completeness_records = []
missing_gap_frames = []

for station_id in station_order:
    prefix = f"{station_id}__"
    station_row_present = joined[f"{prefix}station_id"].notna()
    water_level_present = joined[f"{prefix}water_level"].notna()
    usable_water_level = station_row_present & water_level_present
    imputed_usable = usable_water_level & joined[f"{prefix}imputed"].eq(True)
    weather_complete = (
        usable_water_level
        & joined[f"{prefix}precipitation"].notna()
        & joined[f"{prefix}temperature_2m"].notna()
    )

    completeness_records.append(
        {
            "station_id": station_id,
            "target_hours": target_hours,
            "matched_row_hours": int(station_row_present.sum()),
            "station_row_missing_hours": int((~station_row_present).sum()),
            "water_level_usable_hours": int(usable_water_level.sum()),
            "usable_water_coverage_pct": 100 * usable_water_level.mean(),
            "water_level_missing_hours": int(
                (station_row_present & ~water_level_present).sum()
            ),
            "imputed_usable_hours": int(imputed_usable.sum()),
            "complete_measurement_coverage_pct": 100 * weather_complete.mean(),
            "full_join_qualified": (
                100 * usable_water_level.mean() >= FULL_JOIN_MIN_COVERAGE_PCT
            ),
        }
    )

    missing = ~usable_water_level
    run_id = missing.ne(missing.shift(fill_value=False)).cumsum()
    missing_rows = pd.DataFrame(
        {
            "timestamp": joined["timestamp"],
            "station_row_present": station_row_present,
            "water_level_present": water_level_present,
            "missing": missing,
            "run_id": run_id,
        }
    ).loc[missing]
    if missing_rows.empty:
        continue

    gaps = (
        missing_rows.groupby("run_id", sort=False)
        .agg(
            start=("timestamp", "min"),
            end=("timestamp", "max"),
            missing_hours=("timestamp", "size"),
            station_row_missing_hours=(
                "station_row_present",
                lambda values: (~values).sum(),
            ),
            water_level_missing_hours=(
                "water_level_present",
                lambda values: (~values).sum(),
            ),
        )
        .reset_index(drop=True)
    )
    gaps["station_id"] = station_id
    gaps["gap_type"] = np.select(
        [gaps["start"].eq(target_start), gaps["end"].eq(target_end)],
        ["leading", "trailing"],
        default="internal",
    )
    gaps["missing_reason"] = np.select(
        [
            gaps["station_row_missing_hours"].eq(gaps["missing_hours"]),
            gaps["water_level_missing_hours"].eq(gaps["missing_hours"]),
        ],
        ["station row absent", "water level missing"],
        default="mixed",
    )
    gaps["duration_days"] = gaps["missing_hours"] / 24
    missing_gap_frames.append(
        gaps[
            [
                "station_id",
                "start",
                "end",
                "missing_hours",
                "duration_days",
                "gap_type",
                "missing_reason",
                "station_row_missing_hours",
                "water_level_missing_hours",
            ]
        ]
    )

completeness = pd.DataFrame(completeness_records)
all_missing_gaps = pd.concat(missing_gap_frames, ignore_index=True)
largest_gap_by_station = (
    all_missing_gaps.sort_values("missing_hours", ascending=False)
    .drop_duplicates("station_id")
    .set_index("station_id")
    .loc[station_order]
    .reset_index()
)

display(
    completeness[
        [
            "station_id",
            "matched_row_hours",
            "station_row_missing_hours",
            "water_level_usable_hours",
            "usable_water_coverage_pct",
            "water_level_missing_hours",
            "imputed_usable_hours",
            "complete_measurement_coverage_pct",
            "full_join_qualified",
        ]
    ]
)

display(
    largest_gap_by_station.sort_values("missing_hours", ascending=False)[
        [
            "station_id",
            "start",
            "end",
            "missing_hours",
            "duration_days",
            "gap_type",
            "missing_reason",
        ]
    ]
)

display(
    all_missing_gaps.sort_values("missing_hours", ascending=False).head(20)[
        [
            "station_id",
            "start",
            "end",
            "missing_hours",
            "duration_days",
            "gap_type",
            "missing_reason",
        ]
    ]
)

## 7. Cross-station water-level correlation

Pairwise correlation of hourly water level across the retained stations, using
every timestamp where both stations have an observation (pairwise-complete).
Pearson
captures linear association; Spearman captures monotonic association and is less
sensitive to outlier flood peaks.

In [ ]:
water_wide = pd.DataFrame(
    {station_id: joined[f"{station_id}__water_level"] for station_id in station_order}
)[station_order]
pearson_corr = water_wide.corr(method="pearson")
spearman_corr = water_wide.corr(method="spearman")

correlation_figure = make_subplots(
    rows=1, cols=2, subplot_titles=("Pearson", "Spearman")
)
for column_index, matrix in enumerate([pearson_corr, spearman_corr], start=1):
    correlation_figure.add_trace(
        go.Heatmap(
            z=matrix.to_numpy(),
            x=matrix.columns,
            y=matrix.index,
            zmid=0,
            zmin=-1,
            zmax=1,
            coloraxis="coloraxis",
            hovertemplate="%{y}<br>%{x}<br>r=%{z:.3f}<extra></extra>",
        ),
        row=1,
        col=column_index,
    )
correlation_figure.update_layout(
    title="Pairwise water-level correlation across stations",
    template=PLOT_TEMPLATE,
    height=520,
    width=1_000,
    coloraxis={
        "colorscale": "RdBu",
        "cmin": -1,
        "cmid": 0,
        "cmax": 1,
        "colorbar": {"title": "r"},
    },
)
correlation_figure.show()

## 8. Lag / propagation-delay scan

For each non-target station, its water level is shifted by 0 to
`LAG_SCAN_HOURS` hours and correlated against the target's water level at each
lag. The lag with the strongest correlation is a rough estimate of the time it
takes a change at that station to show up at the target gauge. This uses the
base joined columns directly; no engineered lag or rolling features are involved.

In [ ]:
target_series = joined[f"{TARGET_STATION_ID}__water_level"]
lag_curve_records = []
for station_id in upstream_order:
    series = joined[f"{station_id}__water_level"]
    for lag in range(LAG_SCAN_HOURS + 1):
        lag_curve_records.append(
            {
                "station_id": station_id,
                "lag_hours": lag,
                "correlation": series.shift(lag).corr(target_series),
            }
        )
lag_curves = pd.DataFrame(lag_curve_records)

best_idx = lag_curves.groupby("station_id")["correlation"].idxmax()
lag_summary = lag_curves.loc[best_idx].rename(
    columns={"lag_hours": "best_lag_hours", "correlation": "best_corr"}
)
lag0 = lag_curves.loc[
    lag_curves["lag_hours"] == 0, ["station_id", "correlation"]
].rename(columns={"correlation": "lag0_corr"})
lag_summary = (
    lag_summary.merge(lag0, on="station_id")
    .set_index("station_id")
    .loc[upstream_order]
    .reset_index()[["station_id", "best_lag_hours", "best_corr", "lag0_corr"]]
)
display(lag_summary)

In [ ]:
lag_figure = make_subplots(rows=3, cols=3, subplot_titles=upstream_order)
for position, station_id in enumerate(upstream_order):
    row, col = position // 3 + 1, position % 3 + 1
    subset = lag_curves.loc[lag_curves["station_id"] == station_id]
    lag_figure.add_trace(
        go.Scatter(
            x=subset["lag_hours"],
            y=subset["correlation"],
            mode="lines",
            line={"color": STATION_COLORS[station_id]},
            showlegend=False,
        ),
        row=row,
        col=col,
    )
    best_row = lag_summary.loc[lag_summary["station_id"] == station_id].iloc[0]
    lag_figure.add_trace(
        go.Scatter(
            x=[best_row["best_lag_hours"]],
            y=[best_row["best_corr"]],
            mode="markers",
            marker={"color": "#B22222", "size": 8},
            showlegend=False,
        ),
        row=row,
        col=col,
    )
lag_figure.update_xaxes(title_text="Lag (h)")
lag_figure.update_yaxes(title_text="Pearson r", col=1)
lag_figure.update_layout(
    title=(
        f"Cross-correlation of upstream water level with {TARGET_STATION_ID} "
        f"across 0–{LAG_SCAN_HOURS} h lags"
    ),
    template=PLOT_TEMPLATE,
    height=760,
)
lag_figure.show()

## 9. Upstream-labeling sanity check

The join order labels stations "nearest upstream first" purely by distance along
the river (`positionKm`). This section checks whether that ordering agrees with
what the lag scan actually shows: a station ranked as closer by distance should
usually also rank higher by peak correlation. A station is flagged when its peak
correlation is weak outright, or when its distance rank and correlation rank
disagree by a wide margin — either signals the distance-based label may not
reflect real hydrological upstream influence (a different tributary, a
regulated reach, or a gauge with a very different response profile).

In [ ]:
station_distances = pd.DataFrame(
    {
        "station_id": upstream_order,
        "distance_from_target_km": [distance_to_target(s) for s in upstream_order],
    }
)
sanity = lag_summary.merge(station_distances, on="station_id")
sanity["distance_rank"] = (
    sanity["distance_from_target_km"].rank(method="min").astype(int)
)
sanity["correlation_rank"] = (
    sanity["best_corr"].rank(ascending=False, method="min").astype(int)
)
sanity["rank_gap"] = (sanity["distance_rank"] - sanity["correlation_rank"]).abs()
sanity["flagged"] = (sanity["best_corr"] < WEAK_CORRELATION_THRESHOLD) | (
    sanity["rank_gap"] >= RANK_GAP_FLAG
)
sanity = sanity.sort_values("distance_rank").reset_index(drop=True)
display(
    sanity[
        [
            "station_id",
            "distance_from_target_km",
            "distance_rank",
            "best_corr",
            "best_lag_hours",
            "correlation_rank",
            "rank_gap",
            "flagged",
        ]
    ]
)

## 10. Findings summary

The narrative below is calculated from the validated artifact and the statistical
outputs above. It updates automatically if the inputs are regenerated under the
same contracts.

In [ ]:
strongest_pair = (
    pearson_corr.where(np.triu(np.ones(pearson_corr.shape, dtype=bool), k=1))
    .stack()
    .rename("correlation")
    .reset_index()
    .rename(columns={"level_0": "station_a", "level_1": "station_b"})
    .sort_values("correlation", ascending=False)
    .iloc[0]
)
flagged_stations = sanity.loc[sanity["flagged"], "station_id"].tolist()
if flagged_stations:
    flagged_text = (
        "Flagged, where the distance-based join order disagrees with the lag "
        f"scan: **{', '.join(flagged_stations)}**."
    )
else:
    flagged_text = "No stations were flagged; distance order and lag scan agree."
fastest = lag_summary.sort_values("best_lag_hours").iloc[0]
weakest = lag_summary.sort_values("best_corr").iloc[0]
qualified_stations = completeness.loc[
    completeness["full_join_qualified"], "station_id"
].tolist()
below_threshold_stations = completeness.loc[
    ~completeness["full_join_qualified"], "station_id"
].tolist()
largest_gap = all_missing_gaps.sort_values("missing_hours", ascending=False).iloc[0]


findings = f"""
### Coverage

The joined dataset spans **{len(joined):,} hourly UTC rows** from
**{joined["timestamp"].iloc[0]:%Y-%m-%d %H:%M}** to
**{joined["timestamp"].iloc[-1]:%Y-%m-%d %H:%M}**, with a train/test seam at
**{seam_timestamp:%Y-%m-%d %H:%M} UTC**. Timeline coverage across the
{len(station_order)} stations ranges from **{coverage["coverage_pct"].min():.1f}%**
to **{coverage["coverage_pct"].max():.1f}%**.

### Completeness / full-join eligibility

Using a **{FULL_JOIN_MIN_COVERAGE_PCT:.0f}%** usable water-level coverage
threshold, **{len(qualified_stations)} of {len(station_order)} stations** qualify
for the full-join comparison. Imputed water levels count as usable. Qualified
stations are **{", ".join(qualified_stations)}**; below threshold are
**{", ".join(below_threshold_stations) if below_threshold_stations else "none"}**.
The largest contiguous missing run is at **{largest_gap["station_id"]}**,
from **{largest_gap["start"]:%Y-%m-%d %H:%M}** to
**{largest_gap["end"]:%Y-%m-%d %H:%M} UTC** ({int(largest_gap["missing_hours"]):,} h,
{largest_gap["duration_days"]:.1f} days; {largest_gap["missing_reason"]}).

### Cross-station relationships

The strongest pairwise water-level correlation, excluding self-pairs, is between
**{strongest_pair["station_a"]}** and **{strongest_pair["station_b"]}**
(Pearson r = {strongest_pair["correlation"]:.3f}).

### Lag / propagation delay

Peak lagged correlation with **{TARGET_STATION_ID}** ranges from
**{weakest["best_corr"]:.3f}** ({weakest["station_id"]}) to
**{lag_summary["best_corr"].max():.3f}**. **{fastest["station_id"]}** peaks fastest,
at **{int(fastest["best_lag_hours"])} h**.

### Upstream-labeling sanity check

{flagged_text}
"""
display(Markdown(findings))